# Análisis del caso `n16_d1_s42`

Este notebook analiza el FCM con `uuid = n16_d1_s42`, cargado desde `maps.json`
(`n_concepts = 16`, `density = 1.0`): el **único caso de las 5700 comparaciones**
de `test_lookahead.ipynb` donde el intervalo final de **lookahead** NO queda
contenido en el de **rectified**, con `phi = 0.6`.

Métodos comparados (mismo registry que el experimento):

1. **Naive** — aritmética de intervalos (`interval_state_space`), línea base.
2. **Symbolic** — relajación simbólica concretizada (`symbolic_state_space` + `concretize_state_space`).
3. **Rectified** — intersección acumulada (`rectified_intersection_state_space`, `minimal_warming_iters = 5`).
4. **Lookahead** — ventana completa hasta el final (`k = None`, baseline = Naive).

En cada gráfica, el área azul es el rango del método 1, el área roja el del
método 2 y las líneas negras las 100 simulaciones reales (ground truth).

## Parámetros del experimento

- `activation = "sigmoid"`, `phi = 0.6`, `EPSILON = 1e-6`
- `MAX_ITERS = 50`, `n_samples = 100`, `l_0 = 0`, `u_0 = 1`
- `SEED = 16607`; la matriz `W` se carga de `maps.json` y se usa su transpuesta `Wt`.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
torch.set_grad_enabled(False)
torch.set_default_dtype(dtype)
torch.set_printoptions(precision=30, sci_mode=False)

print(f"Device: {device}")

In [ ]:
from typing import Literal

ActivationFunction = Literal["sigmoid", "tanh"]

In [ ]:
import json

import numpy as np
from core.extras import rescaled_reasoning
from core.methods import (
    interval_state_space,
    symbolic_state_space,
    concretize_state_space,
    rectified_intersection_state_space,
)
from core.metrics import (
    covering,
    is_inside_state,
    is_inside_state_series,
    state_series_from_samples,
)
from core.utils import generate_input_samples, generate_inputs
from experimental.methods import lookahead_state_space
from luis.plot import compare_state_evolutions_and_simulations


MAP_UUID = "n16_d1_s42"
MAX_ITERS = 50

activation: ActivationFunction = "sigmoid"  # "tanh"

phi = 0.6
EPSILON = 1e-6
n_samples = 100
l_0 = 0
u_0 = 1

seed = 16607

print(f"SEED = {seed}")

np.random.seed(seed)
torch.manual_seed(seed)

with open("maps.json", "r") as f:
    samples = json.loads(f.read())

sample = next(s for s in samples if s["uuid"] == MAP_UUID)
n_concepts = sample["n_concepts"]

W = torch.tensor(sample["weigths"], device=device)
Wt = W.T
inputs = generate_inputs(n_concepts, l_0, u_0, device=device).to(dtype=dtype)

naive_states = interval_state_space(
    Wt,
    inputs,
    activation_function=activation,
    phi=phi,
    max_iters=MAX_ITERS,
    device=device,
)

symbolic_concrete_states = concretize_state_space(
    symbolic_state_space(
        Wt,
        inputs,
        activation,
        phi,
        MAX_ITERS,
        overapprox_threshold=None,
        device=device,
    ),
    inputs,
    device=device,
)

rectified_states = rectified_intersection_state_space(
    Wt,
    inputs,
    activation,
    phi,
    MAX_ITERS,
    minimal_warming_iters=5,
)

lookahead_states = lookahead_state_space(
    Wt,
    inputs,
    activation,
    phi,
    k=None,
    max_iters=MAX_ITERS,
    baseline_interval_series=naive_states,
    device=device,
)

A = generate_input_samples(n_samples, n_concepts, l_0, u_0, device=device)

simulated_states = rescaled_reasoning(
    A,
    W,
    T=MAX_ITERS,
    phi=phi,
    transfer_function=torch.sigmoid if activation == "sigmoid" else torch.tanh,
    device=device,
)

sampled_states = state_series_from_samples(simulated_states)

print(f"Samples lie inside naive states: {is_inside_state_series(sampled_states, naive_states, tolerance=EPSILON)}")
print(f"Samples lie inside symbolic states: {is_inside_state_series(sampled_states, symbolic_concrete_states, tolerance=EPSILON)}")
print(f"Samples lie inside rectified states: {is_inside_state_series(sampled_states, rectified_states, tolerance=EPSILON)}")
print(f"Samples lie inside lookahead states: {is_inside_state_series(sampled_states, lookahead_states, tolerance=EPSILON)}")

print()
print(f"Lookahead final interval inside naive final interval: {is_inside_state(lookahead_states[-1], naive_states[-1], tolerance=EPSILON)}")
print(f"Lookahead final interval inside symbolic final interval: {is_inside_state(lookahead_states[-1], symbolic_concrete_states[-1], tolerance=EPSILON)}")
print(f"Lookahead final interval inside rectified final interval: {is_inside_state(lookahead_states[-1], rectified_states[-1], tolerance=EPSILON)}")

print()
print(f"Naive Covering: {covering(naive_states)}")
print(f"Symbolic Covering: {covering(symbolic_concrete_states)}")
print(f"Rectified Covering: {covering(rectified_states)}")
print(f"Lookahead Covering: {covering(lookahead_states)}")

In [ ]:
lookahead_inside_rectified_series = is_inside_state_series(
    lookahead_states, rectified_states, tolerance=EPSILON
)
lookahead_inside_rectified_final = is_inside_state(
    lookahead_states[-1], rectified_states[-1], tolerance=EPSILON
)

print(f"Lookahead series inside rectified series: {lookahead_inside_rectified_series}")
print(f"Lookahead final interval inside rectified final interval: {lookahead_inside_rectified_final}")

violations = ~(
    (rectified_states[:, 0, :] - EPSILON <= lookahead_states[:, 0, :])
    & (lookahead_states[:, 1, :] <= rectified_states[:, 1, :] + EPSILON)
)
t_idx, c_idx = violations.nonzero(as_tuple=True)
print(f"Iterations with violation: {sorted(set(t_idx.tolist()))}")
print(f"Concepts with violation: {sorted(set(c_idx.tolist()))}")

lower_escape = (lookahead_states[:, 0, :] - rectified_states[:, 0, :]).clamp(min=0)
upper_escape = (lookahead_states[:, 1, :] - rectified_states[:, 1, :]).clamp(min=0)
for t, c in sorted(zip(t_idx.tolist(), c_idx.tolist())):
    print(
        f"t={t:>3} C{c + 1}: "
        f"escape inferior={float(lower_escape[t, c]):.6e}, "
        f"escape superior={float(upper_escape[t, c]):.6e}"
    )
print(f"Escape maximo: {float(torch.maximum(lower_escape, upper_escape).max()):.6e}")
print(f"Escape total: {float((lower_escape + upper_escape).sum()):.6e}")

assert not lookahead_inside_rectified_series, (
    "Se esperaba que lookahead NO estuviera contenido en rectified "
    "(caso n16_d1_s42, phi=0.6)"
)

## Evolución de los estados vs simulaciones

Área azul = método 1, área roja = método 2, líneas negras = 100 trayectorias
simuladas. La última celda compara Lookahead (azul) contra Rectified (rojo):
al final de la serie parte del área azul debe escapar de la roja — es la
contención rota detectada por el experimento.

In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    rectified_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Rectified",
)

In [ ]:
compare_state_evolutions_and_simulations(
    naive_states.cpu().numpy(),
    lookahead_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Naive",
    "Lookahead",
)

In [ ]:
compare_state_evolutions_and_simulations(
    lookahead_states.cpu().numpy(),
    rectified_states.cpu().numpy(),
    simulated_states.cpu().numpy(),
    "Lookahead",
    "Rectified",
)